# 22.1 模型持久化:pickle / joblib / ONNX / Model Persistence

**中文**:训练好一个模型只是第一步——要让它**产生价值,必须先把它"存下来"、再在别处"加载"运行**(部署到服务器、嵌入 App、给别的团队用)。这就是**模型持久化(model persistence)**:MLOps 的第一课,也是最容易踩坑的一环。看似简单的"存个模型",背后有三个层次的问题:①**怎么存**(pickle / joblib / ONNX,各有取舍);②**安全**(pickle 能执行任意代码,是真实的攻击面);③**可移植与版本**(sklearn 版本一变,老模型可能加载即崩)。本节用真实代码走通这三层,并揭示一个关键认知:**"模型"本质上只是一堆权重 + 一个计算公式**——理解了这点,你就懂了 ONNX 为什么存在、以及为什么"存对格式"比想象中重要。
**English**: Training a model is only step one — to **create value, you must first "save" it and later "load" and run it elsewhere** (deploy to a server, embed in an app, hand to another team). This is **model persistence**: MLOps's first lesson and one of its easiest traps. The seemingly simple "save a model" hides three layers of problems: ① **how to save** (pickle / joblib / ONNX, each with tradeoffs); ② **security** (pickle can execute arbitrary code — a real attack surface); ③ **portability and versioning** (change the sklearn version and an old model may crash on load). This section walks all three with real code and reveals a key insight: **a "model" is essentially just a bunch of weights + a compute formula** — grasp that and you understand why ONNX exists and why "saving the right format" matters more than you'd think.

---

**中文**:**三种主流持久化方式**:
**English**: **Three mainstream persistence methods**:
- **中文**:**pickle**(Python 标准库):把任意 Python 对象序列化成字节。通用,但**①只能 Python 用 ②不安全(反序列化会执行任意代码)③依赖类定义和库版本**(环境不一致就崩)。
  **pickle** (Python stdlib): serialize any Python object to bytes. Universal, but **① Python-only ② insecure (deserialization executes arbitrary code) ③ depends on class definitions and library versions** (crashes on environment mismatch).
- **中文**:**joblib**:sklearn 生态的首选,本质是**对大 numpy 数组优化过的 pickle**(存得更快更小)。sklearn 模型都用它。但仍有 pickle 的安全/版本问题。
  **joblib**: the sklearn ecosystem's default, essentially **pickle optimized for large numpy arrays** (faster, smaller). All sklearn models use it. But still has pickle's security/version issues.
- **中文**:**ONNX(Open Neural Network Exchange)**:把模型导出成一个**跨框架、跨语言的标准计算图**。它不存"Python 对象",而存"这个模型做的数学运算"。优点:**①任何语言/运行时都能加载(C++/Java/浏览器/手机)②安全(纯数据无代码)③可被推理引擎深度优化(ONNX Runtime 快很多)④解耦训练框架**(用 PyTorch 训、用 ONNX 部署)。
  **ONNX (Open Neural Network Exchange)**: export the model to a **cross-framework, cross-language standard compute graph**. It stores not "Python objects" but "the math operations the model performs." Pros: **① loadable by any language/runtime (C++/Java/browser/phone) ② secure (pure data, no code) ③ deeply optimizable by inference engines (ONNX Runtime is much faster) ④ decouples the training framework** (train in PyTorch, deploy via ONNX).

> 💡 **面试速查 / Interview cheat-sheet（★★ MLOps 基础必考）**
> **中文**:**模型持久化**:①**pickle**(通用但 **Python-only + 不安全(反序列化执行任意代码) + 版本敏感**)；②**joblib**(sklearn 首选, 对大数组优化的 pickle, 更快更小, 仍有 pickle 缺点)；③**ONNX**(跨框架跨语言标准计算图, 安全无代码, 可被 ONNX Runtime 深度优化加速, 训练/部署解耦, 边缘/多语言部署首选)。**关键坑**:①**安全**——绝不 unpickle 不可信文件(=远程代码执行);生产用 ONNX/safetensors 或签名校验。②**版本**——存模型也要**钉住依赖版本**(sklearn/numpy), 版本漂移会加载失败或结果变;存 `requirements.txt`/环境。③**存什么**——不只存模型权重, 还要存**预处理器/pipeline、特征列顺序、版本元数据**(否则线上特征对不上=训练服务偏差)。④大深度模型用 **safetensors**(安全)/框架原生格式。面试金句:*"模型本质是权重+计算图; pickle/joblib 简单但 Python-only、不安全、版本敏感, ONNX 导出标准计算图实现跨语言/安全/可优化的部署; 生产要连预处理一起存、钉住依赖版本、不加载不可信 pickle。"*
> **English**: **Model persistence**: ① **pickle** (universal but **Python-only + insecure (deserialization executes arbitrary code) + version-sensitive**); ② **joblib** (sklearn default, pickle optimized for large arrays, faster/smaller, still pickle's downsides); ③ **ONNX** (cross-framework/cross-language standard compute graph, secure no-code, deeply optimizable by ONNX Runtime, decouples training/deployment, first choice for edge/multi-language). **Key traps**: ① **security** — never unpickle untrusted files (= remote code execution); production uses ONNX/safetensors or signature verification. ② **versioning** — saving a model must **pin dependency versions** (sklearn/numpy); version drift causes load failures or changed results; save `requirements.txt`/environment. ③ **what to save** — not just model weights but also the **preprocessor/pipeline, feature column order, version metadata** (else online features mismatch = training-serving skew). ④ large deep models use **safetensors** (secure) / native framework formats. Interview line: *"A model is essentially weights + a compute graph; pickle/joblib are simple but Python-only, insecure, and version-sensitive, while ONNX exports a standard compute graph for cross-language, secure, optimizable deployment; in production save the preprocessing together, pin dependency versions, and never load untrusted pickles."*


In [ ]:

# ============================================================
# joblib 保存/加载 sklearn 模型 / save & load a sklearn model with joblib
# ============================================================
import numpy as np, joblib, os
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
X,y=make_classification(n_samples=1000, n_features=6, random_state=0)
model=LogisticRegression(max_iter=500).fit(X,y)

joblib.dump(model, "/tmp/model.joblib")                     # 保存 / save
loaded=joblib.load("/tmp/model.joblib")                     # 加载 / load
print("加载后预测与原模型完全一致 / loaded model predictions identical:",
      np.array_equal(model.predict(X), loaded.predict(X)))
print(f"joblib 文件大小 / file size: {os.path.getsize('/tmp/model.joblib')} bytes")


In [ ]:

# ============================================================
# 揭示本质:模型 = 权重 + 公式(所以能跨框架/语言部署)/ a model = weights + formula
# 中文:逻辑回归"模型"其实只有 coef_(权重)+ intercept_(偏置)+ sigmoid 公式。我们从模型里抽出这些参数,
#      用纯 numpy 重写前向计算——不依赖 sklearn 也能得到一模一样的预测。这正是 ONNX 的思想:序列化"计算图"而非"Python 对象"。
# English: a logistic-regression "model" is really just coef_ (weights) + intercept_ (bias) + the sigmoid formula.
#      Extract those params and reimplement the forward pass in pure numpy — identical predictions without sklearn. This IS ONNX's idea.
# ============================================================
w=model.coef_[0]; b=model.intercept_[0]                     # 抽出参数 / extract parameters
sigmoid=lambda z: 1/(1+np.exp(-z))
manual_proba=sigmoid(X@w + b)                               # 纯 numpy 前向 / pure-numpy forward pass
print("纯 numpy 复现与 sklearn 概率一致 / pure-numpy matches sklearn proba:",
      np.allclose(manual_proba, model.predict_proba(X)[:,1]))
print("→ '模型'就是这些数字 + 这个公式:")
print("   weights[:3] =", w[:3].round(3), " bias =", round(float(b),3))
print("→ ONNX 就是把这个'计算图'(而非 Python 对象)存成跨语言标准格式, 于是 C++/浏览器/手机都能加载运行")


In [ ]:

# ============================================================
# 三个致命坑的演示 / the three fatal traps
# ============================================================
import pickle, sklearn
# ① 安全:pickle 反序列化会执行任意代码(演示原理, 不加载恶意文件)/ SECURITY: pickle can run arbitrary code on load
class Evil:
    def __reduce__(self): return (print, ("[演示] 加载这个 pickle 时, 任意代码被执行了! (真实攻击会 rm -rf / 或窃取密钥)",))
malicious=pickle.dumps(Evil())
print("① 安全坑演示 / security demo:")
pickle.loads(malicious)                                     # 仅打印, 但真实攻击可执行任何命令 / just prints; real attacks run anything
print("   → 绝不 pickle.load 不可信来源的文件!生产用 ONNX/safetensors 或签名校验\n")

# ② 版本:保存的模型带着它训练时的库版本, 版本漂移会出问题 / VERSION: models carry their training-time lib version
print(f"② 版本坑 / version trap: 本模型用 sklearn {sklearn.__version__} 训练。")
print("   若部署环境 sklearn 版本不同, 可能加载失败或(更危险)静默给出不同结果 → 必须钉住依赖版本\n")

# ③ 存什么:光存模型不够, 要连预处理一起存(否则训练-服务偏差)/ WHAT TO SAVE: save the preprocessor too
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipe=Pipeline([("scaler",StandardScaler()), ("clf",LogisticRegression(max_iter=500))]).fit(X,y)
joblib.dump(pipe, "/tmp/pipeline.joblib")                   # 存整条 pipeline, 预处理+模型一体 / save the whole pipeline
print("③ 存什么 / what to save: 保存整条 Pipeline(预处理+模型)而非裸模型——")
print("   否则线上要手动复刻标准化, 极易训练-服务偏差(training-serving skew)。加载即用:")
print("   pipeline 预测前3个 / pipeline preds[:3]:", joblib.load("/tmp/pipeline.joblib").predict(X[:3]))


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **"模型"没有你想的那么神秘——它就是权重 + 计算图**:我们把逻辑回归拆开,发现它不过是 `coef_`、`intercept_` 加一个 sigmoid 公式,用纯 numpy 几行就完美复现。**任何模型(哪怕是 GPT)本质都是"一堆数字参数 + 一个确定的计算流程"**。想通这一点,持久化的所有格式之争就清晰了:pickle/joblib 存的是"Python 对象"(所以 Python 专属、不安全、版本敏感),而 ONNX 存的是"计算图本身"(所以跨语言、安全、可优化)。**部署的本质,就是把"权重 + 计算图"搬到另一个环境正确地跑起来。**
2. **pickle 的安全问题是真实且严重的,不是理论**:我们演示了一个 `__reduce__` 就能让"加载模型"变成"执行任意代码"——真实攻击里这可以是删库、窃取云凭证、植入后门。所以**永远不要 `pickle.load` / `joblib.load` 来路不明的文件**(从网上下载的、别人发来的 `.pkl`)。这也是为什么生产环境越来越倾向 **ONNX、safetensors** 这类"纯数据、无代码"的格式,或对模型文件做签名校验。很多人从没意识到"加载一个模型"是一个攻击面。
3. **持久化最大的坑不是"怎么存",而是"训练-服务偏差"和"版本漂移"**:①**训练-服务偏差(training-serving skew)**——如果你只存了模型、却在部署时手动重写预处理(标准化、编码),两边逻辑稍有不同,线上预测就会错得莫名其妙。**正确做法:把预处理和模型打包成一个 Pipeline 一起存**(我们演示了),线上加载即用,保证训练和服务走完全相同的变换。②**版本漂移**——模型文件"记得"它训练时的 sklearn/numpy 版本,环境不一致轻则加载报错、重则**静默给出不同结果**(最可怕的 bug)。所以生产必须**连同依赖版本一起固化**(requirements.txt/容器镜像),并存好模型的元数据(训练时间、数据版本、指标)。**结论:模型持久化的专业性不在于会调 `joblib.dump`,而在于把'预处理+模型+依赖版本+元数据'当成一个整体来管理——这正是 MLOps 与'只会训模型'的分水岭。**

**English**:
1. **A "model" is less mysterious than you think — it's just weights + a compute graph**: we opened up logistic regression and found merely `coef_`, `intercept_`, and a sigmoid formula, perfectly reproduced in a few lines of pure numpy. **Any model (even GPT) is essentially "a bunch of numeric parameters + a fixed computation flow."** Grasp this and all the format debates clarify: pickle/joblib store "Python objects" (hence Python-only, insecure, version-sensitive), while ONNX stores "the compute graph itself" (hence cross-language, secure, optimizable). **Deployment is essentially moving "weights + compute graph" to another environment and running it correctly.**
2. **pickle's security problem is real and serious, not theoretical**: we showed how one `__reduce__` turns "load a model" into "execute arbitrary code" — in real attacks this could delete databases, steal cloud credentials, or plant backdoors. So **never `pickle.load` / `joblib.load` files of unknown origin** (a `.pkl` downloaded from the web or sent by someone). This is why production increasingly favors "pure-data, no-code" formats like **ONNX, safetensors**, or signature verification on model files. Many never realize that "loading a model" is an attack surface.
3. **Persistence's biggest trap isn't "how to save" but "training-serving skew" and "version drift"**: ① **training-serving skew** — if you save only the model but rewrite preprocessing (scaling, encoding) by hand at deployment, any slight logic difference makes online predictions inexplicably wrong. **The right way: package preprocessing and model into one Pipeline and save together** (as we showed), so online loading uses the exact same transforms as training. ② **version drift** — a model file "remembers" its training-time sklearn/numpy version, and environment mismatch causes load errors at best, or at worst **silently different results** (the scariest bug). So production must **freeze dependency versions together** (requirements.txt/container image) and store model metadata (training time, data version, metrics). **Conclusion: the professionalism of model persistence isn't knowing `joblib.dump` but managing "preprocessing + model + dependency versions + metadata" as one whole — precisely the divide between MLOps and "only trains models."**

> 💼 **实战视角 / Practical angle**
> **中文**:模型持久化落地:①**存整条 Pipeline**(预处理+模型)而非裸模型, 杜绝训练-服务偏差;②**钉住依赖**——用容器镜像或 `requirements.txt` 固化 sklearn/numpy/python 版本, 存模型元数据(版本、训练数据、指标、时间);③**安全**——不 `load` 不可信文件, 深度模型用 **safetensors**(HuggingFace 默认, 无代码执行), 跨语言/边缘部署用 **ONNX**(+ONNX Runtime 加速);④**版本化模型**——用 MLflow Model Registry(22.8)或对象存储 + 语义版本管理, 支持回滚;⑤ONNX 导出:sklearn 用 `skl2onnx`、PyTorch 用 `torch.onnx.export`。面试金句:*"模型=权重+计算图; sklearn 用 joblib(优化的 pickle)但 Python-only/不安全/版本敏感; 生产要存整条 Pipeline(防训练-服务偏差)、钉依赖版本、存元数据、不加载不可信 pickle; 跨语言/边缘/加速部署导出 ONNX, 大模型用 safetensors。"*
> **English**: Model persistence in practice: ① **save the whole Pipeline** (preprocessing + model), not the bare model, to eliminate training-serving skew; ② **pin dependencies** — freeze sklearn/numpy/python versions via a container image or `requirements.txt`, and store model metadata (version, training data, metrics, time); ③ **security** — don't `load` untrusted files; use **safetensors** for deep models (HuggingFace default, no code execution) and **ONNX** for cross-language/edge (+ ONNX Runtime acceleration); ④ **version models** — via MLflow Model Registry (22.8) or object storage + semantic versioning, enabling rollback; ⑤ ONNX export: `skl2onnx` for sklearn, `torch.onnx.export` for PyTorch. Interview line: *"A model = weights + compute graph; sklearn uses joblib (optimized pickle) but it's Python-only/insecure/version-sensitive; production should save the whole Pipeline (prevent training-serving skew), pin dependency versions, store metadata, and never load untrusted pickles; export ONNX for cross-language/edge/accelerated deployment, and safetensors for large models."*

---
### 小结 / Summary
- **中文**:模型=权重+计算图; pickle(通用但 Python-only/不安全/版本敏感)、joblib(sklearn 首选)、ONNX(跨语言/安全/可优化)。
- **English**: Model = weights + compute graph; pickle (universal but Python-only/insecure/version-sensitive), joblib (sklearn default), ONNX (cross-language/secure/optimizable).
- **中文**:三大坑:安全(不加载不可信 pickle)、版本漂移(钉依赖)、训练-服务偏差(存整条 Pipeline)。
- **English**: Three traps: security (don't load untrusted pickles), version drift (pin dependencies), training-serving skew (save the whole Pipeline).
- **中文**:生产存 Pipeline+依赖版本+元数据; 跨语言/边缘用 ONNX, 大模型用 safetensors, 模型版本化支持回滚。
- **English**: Production saves Pipeline + dependency versions + metadata; ONNX for cross-language/edge, safetensors for large models, version models for rollback.
